# 08 · 官方算例复现

DESY 官方算例一键运行并与黄金样本比对 (Manual/Aperture/Wake/Cavity/Curved_Cathode/90deg_bend/Plasma_1/Plasma_2)。

In [ ]:
%run _bootstrap.py

In [ ]:
from pathlib import Path
EXAMPLES_DIR = PROJECT_ROOT / "examples"
for d in sorted(EXAMPLES_DIR.iterdir()):
    if d.is_dir():
        files = [f.name for f in d.iterdir() if f.is_file()][:6]
        print("%-22s %s" % (d.name, ", ".join(files)))

In [ ]:
import shutil
EXAMPLE = "Manual_Example"
src = EXAMPLES_DIR / EXAMPLE
work = SIM_DIR / EXAMPLE
work.mkdir(parents=True, exist_ok=True)
import re
for f in src.iterdir():
    if f.is_file() and f.suffix in (".in", ".dat", ".ini"):
        target = "astra.in" if f.name == "Example.in" else f.name
        shutil.copy2(f, work / target)
# 修正输入卡中的 Distribution 绝对路径 (官方算例里的路径是发布机器的)
deck = (work / "astra.in").read_text()
deck = re.sub(r"Distribution\s*=\s*'[^']*'", "Distribution='" + str(work / "Example.ini") + "'", deck)
(work / "astra.in").write_text(deck)
print("已复制到:", work)

In [ ]:
from astra_tools.run import run_program
run_program(GENERATOR_EXE, work, input_file="generator.in")
run_program(ASTRA_EXE, work, input_file="astra.in")

In [ ]:
# 与黄金样本比对 (发射度/束斑, 容差 0.5%)
import numpy as np
from astra_tools.io.astra_emit import parse_output_file
new = parse_output_file(work / "astra.Xemit.001")
golden = parse_output_file(src / "Example.Xemit.001")
for key in ("norm_emit_x", "sigma_x"):
    a, b = float(np.asarray(new[key])[-1]), float(np.asarray(golden[key])[-1])
    rel = abs(a - b) / abs(b) * 100
    print("%-14s new=%.6g golden=%.6g rel=%.4f%% %s" % (key, a, b, rel, "OK" if rel < 0.5 else "MISMATCH"))